In [17]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xgrads import open_CtlDataset

ROOT = Path("../data/bc/t30")

print("Working dir:", Path.cwd())
print("Input root exists:", ROOT.exists())

Working dir: /leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/run
Input root exists: True


In [22]:
for ctl in ROOT.glob("*/*.ctl"):
    print("=" * 80)
    print(ctl.name)

    ds = open_CtlDataset(str(ctl))   

noaa_anom_1854_2010_mean1979_2008.t30.ctl
orog_lsm_alb.t30.ctl
surfv_st3_7908clim.t30.land.ctl
veget.t30.land.ctl
surfv_st4_7908clim.t30.land.ctl
sndep_7908clim.t30.land.ctl
soilw_7908clim.t30.land.ctl
seaice_7908clim.t30.sea.ctl
surfv_st2_7908clim.t30.land.ctl
sst_7908clim.t30.sea.ctl
surfv_st1_7908clim.t30.land.ctl


In [24]:
def read_ctl_text(ctl):
    return Path(ctl).read_text(errors="ignore").splitlines()


def get_ctl_line(lines, key):
    key = key.lower()
    for line in lines:
        if line.strip().lower().startswith(key):
            return line.strip()
    return None


def parse_tdef(tdef_line):
    if tdef_line is None:
        return {}

    parts = tdef_line.split()
    out = {
        "tdef_raw": tdef_line,
        "nt": None,
        "t_start": None,
        "t_step": None,
    }

    if len(parts) >= 5:
        out["nt"] = int(parts[1])
        out["t_start"] = parts[3]
        out["t_step"] = parts[4]

    return out


def infer_years_from_name(name):
    """
    Пытается угадать годы из имени файла:
    7908clim -> 1979-2008
    1900_1991 / 1900-1991 -> 1900-1991
    """
    name = str(name)

    m = re.search(r"(19\d{2}|20\d{2})\D+(19\d{2}|20\d{2})", name)
    if m:
        return int(m.group(1)), int(m.group(2)), "from filename 4-digit"

    m = re.search(r"(?<!\d)(\d{2})(\d{2})clim", name.lower())
    if m:
        y1 = int(m.group(1))
        y2 = int(m.group(2))

        # для климатологий типа 7908: 79 -> 1979, 08 -> 2008
        start = 1900 + y1 if y1 >= 30 else 2000 + y1
        end = 1900 + y2 if y2 >= 30 else 2000 + y2

        if end < start:
            end += 100

        return start, end, "from filename yy-yyclim"

    return None, None, ""


def parse_vars(lines):
    vars_list = []
    for i, line in enumerate(lines):
        if line.strip().lower().startswith("vars"):
            nvars = int(line.split()[1])
            for j in range(i + 1, i + 1 + nvars):
                parts = lines[j].split()
                if len(parts) >= 2:
                    name = parts[0]
                    zlevs = parts[1]
                    desc = " ".join(parts[3:]) if len(parts) > 3 else ""
                    vars_list.append((name, zlevs, desc))
            break
    return vars_list

In [25]:
rows = []

for ctl in sorted(ROOT.glob("*/*.ctl")):
    lines = read_ctl_text(ctl)

    dset = get_ctl_line(lines, "DSET")
    title = get_ctl_line(lines, "TITLE")
    options = get_ctl_line(lines, "OPTIONS")
    tdef = get_ctl_line(lines, "TDEF")
    vars_list = parse_vars(lines)

    tinfo = parse_tdef(tdef)
    y_start, y_end, year_source = infer_years_from_name(ctl.name)

    try:
        ds = open_CtlDataset(str(ctl))
        xgrads_time0 = str(ds.time.values[0]) if "time" in ds.coords else ""
        xgrads_time1 = str(ds.time.values[-1]) if "time" in ds.coords else ""
    except Exception as e:
        xgrads_time0 = ""
        xgrads_time1 = ""
    
    rows.append({
        "folder": ctl.parent.name,
        "ctl": ctl.name,
        "path": str(ctl),
        "title": title,
        "dset": dset,
        "options": options,
        "nt": tinfo.get("nt"),
        "t_start_ctl": tinfo.get("t_start"),
        "t_step_ctl": tinfo.get("t_step"),
        "time0_xgrads": xgrads_time0,
        "time1_xgrads": xgrads_time1,
        "year_start_guess": y_start,
        "year_end_guess": y_end,
        "year_source": year_source,
        "variables": ", ".join(v[0] for v in vars_list),
    })

forcing_inventory = pd.DataFrame(rows)
forcing_inventory

,folder,ctl,path,title,dset,options,nt,t_start_ctl,t_step_ctl,time0_xgrads,time1_xgrads,year_start_guess,year_end_guess,year_source,variables
0,anom,noaa_anom_1854_2010_mean1979_2008.t30.ctl,../data/bc/t30/anom/noaa_anom_1854_2010_mean19...,TITLE SST at t30 from REYNOLDS (1874-2010...,DSET ^noaa_anom_1854_2010_mean1979_2008.t3...,OPTIONS SEQUENTIAL YREV big_endian,1884,1jan1854,1mo,1854-01-01T00:00:00.000000000,2010-12-01T00:00:00.000000000,2010.0,1979.0,from filename 4-digit,SSTA
1,clim,orog_lsm_alb.t30.ctl,../data/bc/t30/clim/orog_lsm_alb.t30.ctl,"TITLE orography, land-sea mask and albedo ...",DSET ^orog_lsm_alb.t30.grd,OPTIONS SEQUENTIAL YREV big_endian,1,1jan1981,1mo,1981-01-01T00:00:00.000000000,1981-01-01T00:00:00.000000000,NaN,NaN,,"GH, LSM, ALB"
2,clim,seaice_7908clim.t30.sea.ctl,../data/bc/t30/clim/seaice_7908clim.t30.sea.ctl,TITLE Sea-ice fraction at T30 gg,DSET ^seaice_7908clim.t30.sea.grd,OPTIONS SEQUENTIAL YREV big_endian,12,1jan1981,1mo,1981-01-01T00:00:00.000000000,1981-12-01T00:00:00.000000000,1979.0,2008.0,from filename yy-yyclim,SICE
3,clim,sndep_7908clim.t30.land.ctl,../data/bc/t30/clim/sndep_7908clim.t30.land.ctl,TITLE Snow depth climatology at T30 gg,DSET ^sndep_7908clim.t30.land.grd,OPTIONS SEQUENTIAL YREV big_endian,12,1jan1981,1mo,1981-01-01T00:00:00.000000000,1981-12-01T00:00:00.000000000,1979.0,2008.0,from filename yy-yyclim,SNOWD
4,clim,soilw_7908clim.t30.land.ctl,../data/bc/t30/clim/soilw_7908clim.t30.land.ctl,TITLE Soil wetness on T30 gg,DSET ^soilw_7908clim.t30.land.grd,OPTIONS SEQUENTIAL YREV big_endian,12,1jan1981,1mo,1981-01-01T00:00:00.000000000,1981-12-01T00:00:00.000000000,1979.0,2008.0,from filename yy-yyclim,"SWL1, SWL2, SWL3"
5,clim,sst_7908clim.t30.sea.ctl,../data/bc/t30/clim/sst_7908clim.t30.sea.ctl,TITLE Sea surface temperature on T30 gg,DSET ^sst_7908clim.t30.sea.grd,OPTIONS SEQUENTIAL YREV big_endian,12,1jan1981,1mo,1981-01-01T00:00:00.000000000,1981-12-01T00:00:00.000000000,1979.0,2008.0,from filename yy-yyclim,ST
6,clim,surfv_st1_7908clim.t30.land.ctl,../data/bc/t30/clim/surfv_st1_7908clim.t30.lan...,TITLE Land skin temperature on T30 gg,DSET ^surfv_st1_7908clim.t30.land.grd,OPTIONS SEQUENTIAL YREV big_endian,12,1jan1981,1mo,1981-01-01T00:00:00.000000000,1981-12-01T00:00:00.000000000,1979.0,2008.0,from filename yy-yyclim,SKT
7,clim,surfv_st2_7908clim.t30.land.ctl,../data/bc/t30/clim/surfv_st2_7908clim.t30.lan...,TITLE Land skin temperature on T30 gg,DSET ^surfv_st2_7908clim.t30.land.grd,OPTIONS SEQUENTIAL YREV big_endian,12,1jan1981,1mo,1981-01-01T00:00:00.000000000,1981-12-01T00:00:00.000000000,1979.0,2008.0,from filename yy-yyclim,SKT
8,clim,surfv_st3_7908clim.t30.land.ctl,../data/bc/t30/clim/surfv_st3_7908clim.t30.lan...,TITLE Land skin temperature on T30 gg,DSET ^surfv_st3_7908clim.t30.land.grd,OPTIONS SEQUENTIAL YREV big_endian,12,1jan1981,1mo,1981-01-01T00:00:00.000000000,1981-12-01T00:00:00.000000000,1979.0,2008.0,from filename yy-yyclim,SKT
9,clim,surfv_st4_7908clim.t30.land.ctl,../data/bc/t30/clim/surfv_st4_7908clim.t30.lan...,TITLE Land skin temperature on T30 gg,DSET ^surfv_st4_7908clim.t30.land.grd,OPTIONS SEQUENTIAL YREV big_endian,12,1jan1981,1mo,1981-01-01T00:00:00.000000000,1981-12-01T00:00:00.000000000,1979.0,2008.0,from filename yy-yyclim,SKT
